# 09 — Transactions & Concurrency

## 1. Why this concept exists

A data system is rarely used by one operation at a time. Multiple readers and writers can overlap, failures can happen halfway through a workflow, and a process can crash after some statements have committed.

Transactions provide a controlled unit of work.

## 2. Short mental model

A useful starting point is **ACID**:

- **Atomicity** — the transaction commits as a unit or rolls back.
- **Consistency** — committed data satisfies the database's declared rules.
- **Isolation** — concurrent transactions are protected from certain forms of interference.
- **Durability** — committed changes survive normal process/database restart.

Isolation is not one magical level. It is a set of guarantees about what concurrent transactions can observe.

This notebook uses SQLite and two connections to a file-backed database. It is a real concurrency experiment, not a distributed-database simulation.

## Capability contract

**Capability:** C02 — Data Access Querying & Modeling  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Translate a business question into correct data access and defend the model/query behavior.


## 🖊️ Sketch note

![Sketch note — Transactions & Concurrency](../assets/sketch-notes/09-transactions-concurrency.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S07 — Query plans and indexing (DATA-1102)** → [`../work_simulations/07_acme_commerce_latency_budget/README.md`](../work_simulations/07_acme_commerce_latency_budget/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Learning objectives

- explain atomic commit;
- distinguish a transaction failure from a concurrency anomaly;
- reproduce one controlled interleaving;
- state the guarantee a production system needs.


## Predict before you run

Predict which value a concurrent reader should observe under the chosen isolation behavior.


## Measure

Record the observed state transitions, not only the final value.


In [ ]:
import sqlite3
import tempfile
from pathlib import Path

db_path = Path(tempfile.gettempdir()) / "transactions_concurrency_lab.db"
if db_path.exists():
    db_path.unlink()

def connect():
    return sqlite3.connect(
        db_path, timeout=0.5, isolation_level=None, check_same_thread=False
    )

con = connect()
con.execute("PRAGMA journal_mode=WAL")
con.execute("CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, balance INTEGER NOT NULL CHECK(balance >= 0))")
con.execute("INSERT INTO accounts VALUES (1, 100)")
con.execute("INSERT INTO accounts VALUES (2, 100)")
con.commit()
con.close()

print("Initial balances:", sqlite3.connect(db_path).execute(
    "SELECT * FROM accounts ORDER BY account_id"
).fetchall())

## 3. Atomicity: all-or-nothing

Suppose a transfer debits one account and credits another. If the second statement fails, the first should not remain committed.

We deliberately violate a `CHECK` constraint halfway through a transaction and roll back.

In [ ]:
con = connect()
con.execute("BEGIN")
try:
    con.execute("UPDATE accounts SET balance = balance - 30 WHERE account_id = 1")
    con.execute("UPDATE accounts SET balance = balance - 1000 WHERE account_id = 2")
    con.commit()
except sqlite3.IntegrityError as exc:
    print("Expected failure:", exc)
    con.rollback()

print("Balances after rollback:",
      con.execute("SELECT * FROM accounts ORDER BY account_id").fetchall())
assert con.execute("SELECT balance FROM accounts WHERE account_id=1").fetchone()[0] == 100
con.close()

## 4. A successful transaction

Now perform the same kind of transfer with valid amounts.

The two updates become one atomic unit from the database's point of view.

In [ ]:
con = connect()
con.execute("BEGIN")
con.execute("UPDATE accounts SET balance = balance - 30 WHERE account_id = 1")
con.execute("UPDATE accounts SET balance = balance + 30 WHERE account_id = 2")
con.commit()

balances = con.execute("SELECT * FROM accounts ORDER BY account_id").fetchall()
print("Balances after commit:", balances)
assert balances == [(1, 70), (2, 130)]
con.close()

## 5. Concurrency: readers and writers

SQLite's WAL mode allows readers and a writer to make useful progress concurrently.

We will hold a writer transaction open while another connection reads the database. The reader should continue seeing the last committed state rather than the writer's uncommitted change.

In [ ]:
writer = connect()
reader = connect()

writer.execute("BEGIN IMMEDIATE")
writer.execute("UPDATE accounts SET balance = balance - 10 WHERE account_id = 1")

writer_view = writer.execute("SELECT balance FROM accounts WHERE account_id=1").fetchone()[0]
reader_view = reader.execute("SELECT balance FROM accounts WHERE account_id=1").fetchone()[0]

print("Writer sees:", writer_view)
print("Reader sees:", reader_view)

assert writer_view == 60
assert reader_view == 70

writer.rollback()
writer.close()
reader.close()

## 6. Break it deliberately: lock contention

Transactions can also block one another.

The experiment below starts a write transaction, then attempts another write from a second connection with a short timeout. The second writer should fail quickly with a locking error.

In [ ]:
writer1 = connect()
writer2 = sqlite3.connect(db_path, timeout=0.1, isolation_level=None, check_same_thread=False)

writer1.execute("BEGIN IMMEDIATE")
writer1.execute("UPDATE accounts SET balance = balance + 1 WHERE account_id = 1")

try:
    writer2.execute("BEGIN IMMEDIATE")
    writer2.execute("UPDATE accounts SET balance = balance + 1 WHERE account_id = 2")
    writer2.commit()
    print("Unexpectedly acquired the write lock.")
except sqlite3.OperationalError as exc:
    print("Expected contention:", exc)
    writer2.rollback()

writer1.rollback()
writer1.close()
writer2.close()

## Your task

**Goal:** Make a money transfer atomic.

**Do this:** Implement `transfer(from_id, to_id, amount)` with validation and one transaction. Prove that a failure rolls back and total balance is preserved.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail

## 7. Diagnose what happened

When debugging transactional behavior, separate these questions:

- **What was committed?**
- **What was still uncommitted?**
- **Which connection was reading?**
- **Which lock or isolation behavior was in effect?**
- **Could the operation be retried safely?**

A transaction is not a replacement for application-level correctness. Idempotency, constraints, retry behavior, timeouts and deadlock/lock handling still matter.

**Engineering challenge:** implement a `transfer(from_id, to_id, amount)` function that validates the amount, updates both accounts in one transaction, rolls back on any error, and leaves the total balance unchanged.


## 8. Production implications

Real systems add more complexity:

- concurrent transactions across many processes;
- connection pools;
- lock contention and timeouts;
- isolation-level choices;
- deadlocks in databases that support them;
- retries and idempotency;
- durable logging/recovery;
- distributed transactions or sagas across services.

The production habit is:

> **Define the consistency guarantee you need, then test the failure and concurrency cases that could violate it.**

In [ ]:
con = connect()
total = con.execute("SELECT SUM(balance) FROM accounts").fetchone()[0]
print("Final total balance:", total)
assert total == 200
con.close()
print("Invariant passed: failed transactions did not change committed total balance.")

---

## Check your work

You have finished the exercise. **Now** compare your reasoning and evidence with the reference solution.

[Open the reference solution for Notebook 09](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/09.md)
